# Character Set tagger

Tags the pictures you drop on the website's **Character Set** tab. It does **not** train a LoRA.

1. **Runtime → Change runtime type → T4** (free is enough).
2. Run **cell 1**, then **cell 2**.
3. Copy the `READY:` URL into the **Character Set** tab.

The first run downloads the tagger (~400 MB). Leave this tab open while you drop pictures. When you are done: **Runtime → Disconnect and delete runtime**.



In [ ]:
# Cell 1: the tagger only (no training, no Google Drive)
import os, subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "onnxruntime-gpu", "huggingface_hub", "fastapi", "uvicorn", "pillow", "numpy", "nest_asyncio"])
# Colab's free T4 sometimes has no CUDA onnxruntime build; fall back to CPU (still fast for a few faces)
try:
    import onnxruntime as ort
    assert "CUDAExecutionProvider" in ort.get_available_providers() or True
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"])

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.check_call(["wget", "-q", "-O", "/usr/local/bin/cloudflared",
                           "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
    os.chmod("/usr/local/bin/cloudflared", 0o755)

from huggingface_hub import hf_hub_download
TAGGER_REPO = "SmilingWolf/wd-swinv2-tagger-v3"
MODEL_PATH = hf_hub_download(TAGGER_REPO, "model.onnx")
TAGS_PATH = hf_hub_download(TAGGER_REPO, "selected_tags.csv")
print("Tagger ready.", MODEL_PATH)



In [ ]:
# Cell 2: tagger server. Paste the READY: URL into the Character Set tab.
if "MODEL_PATH" not in globals():
    raise RuntimeError("Run cell 1 first, then run this cell again.")

import base64, csv, io, os, re, subprocess, threading, time
import nest_asyncio, numpy as np, onnxruntime as ort, uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from pydantic import BaseModel

nest_asyncio.apply()
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
SESSION = ort.InferenceSession(MODEL_PATH, providers=providers)
INPUT_NAME = SESSION.get_inputs()[0].name
SHAPE = SESSION.get_inputs()[0].shape
# SwinV2 v3 is NHWC [1, 448, 448, 3]. Some builds are NCHW [1, 3, 448, 448].
NCHW = len(SHAPE) == 4 and SHAPE[1] == 3
INPUT_SIZE = 448
# One entry per model output, same order as selected_tags.csv. Category 9 is the rating, not an emotion.
TAGS = []
with open(TAGS_PATH, newline="", encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        TAGS.append((row["name"].strip(), str(row.get("category", "0"))))

ROOT = "/content/character_set_tags"
os.makedirs(ROOT, exist_ok=True)
STATE = {"status": "idle", "task": None, "phase": "", "error": None, "result": None}
LOCK = threading.Lock()

def clean_name(name):
    name = re.sub(r"[^\w-]+", "_", (name or "").strip()).strip("_")
    if not name:
        raise ValueError("Type a name first")
    return name

def clean_file(name):
    name = os.path.basename(name or "")
    if not name or name != re.sub(r"[^\w.\-]+", "_", name):
        raise ValueError("Bad file name")
    return name

def folder(name):
    d = os.path.join(ROOT, clean_name(name))
    os.makedirs(d, exist_ok=True)
    return d

def images_of(name):
    d = folder(name)
    return sorted(f for f in os.listdir(d) if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".bmp")))

def caption_path(name, file):
    return os.path.join(folder(name), os.path.splitext(file)[0] + ".txt")

def read_caption(name, file):
    p = caption_path(name, file)
    return open(p, encoding="utf-8").read().strip() if os.path.exists(p) else ""

def write_caption(name, file, text):
    open(caption_path(name, file), "w", encoding="utf-8").write(text or "")

def dataset_payload(name):
    return {"name": clean_name(name),
            "images": [{"file": f, "caption": read_caption(name, f)} for f in images_of(name)]}

def tag_image(path, threshold):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), (255, 255, 255))
    canvas.paste(img, ((side - w) // 2, (side - h) // 2))
    canvas = canvas.resize((int(INPUT_SIZE), int(INPUT_SIZE)), Image.BICUBIC)
    arr = np.asarray(canvas, dtype=np.float32)[:, :, ::-1]  # BGR, 0-255
    if NCHW:
        arr = np.transpose(arr, (2, 0, 1))
    probs = SESSION.run(None, {INPUT_NAME: arr[None, ...]})[0][0]
    picked = []
    n = min(len(probs), len(TAGS))
    for i in range(n):
        name, cat = TAGS[i]
        if cat == "9" or float(probs[i]) < threshold:
            continue
        picked.append(name.replace(" ", "_"))
    return picked

def _begin(task, phase):
    with LOCK:
        if STATE["status"] == "running":
            raise RuntimeError("The tagger is already busy")
        STATE.update(status="running", task=task, phase=phase, error=None, result=None)

def _finish(result=None, error=None):
    STATE.update(status="error" if error else "done", phase="", error=str(error) if error else None, result=result)

def _worker(name, threshold, mode):
    try:
        files = images_of(name)
        if not files:
            raise RuntimeError("No images to tag")
        STATE["phase"] = "Tagging"
        changed = 0
        for f in files:
            new_tags = tag_image(os.path.join(folder(name), f), threshold)
            old = [t.strip() for t in read_caption(name, f).split(",") if t.strip()] if mode == "append" else []
            have = {t.lower() for t in old}
            for t in new_tags:
                if t.lower() not in have:
                    old.append(t)
                    have.add(t.lower())
            write_caption(name, f, ", ".join(old))
            changed += 1
        _finish(result={"tagged": changed})
    except Exception as e:
        _finish(error=e)

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"])

def fail(msg, code=400):
    raise HTTPException(status_code=code, detail=str(msg))

class Upload(BaseModel):
    filename: str
    data: str
    caption: str | None = None

class AutoTag(BaseModel):
    threshold: float = 0.35
    mode: str = "replace"
    trigger_word: str = ""
    exclude_tags: str = ""

@app.get("/training/info")
def info():
    return {"mode": "character_set_tagger"}

@app.get("/training/status")
def status():
    return dict(STATE)

@app.get("/training/dataset/{name}")
def get_dataset(name: str):
    try:
        return dataset_payload(name)
    except ValueError as e:
        fail(e)

@app.post("/training/dataset/{name}/upload")
def upload(name: str, req: Upload):
    try:
        d = folder(name)
        file = clean_file(req.filename)
    except ValueError as e:
        fail(e)
    if not file.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".bmp")):
        fail("Only jpg, png, webp and bmp")
    raw = base64.b64decode(re.sub(r"^data:[^,]*,", "", req.data))
    try:
        Image.open(io.BytesIO(raw)).verify()
    except Exception:
        fail("That file is not a readable image")
    stem, ext = os.path.splitext(file)
    i = 1
    while os.path.exists(os.path.join(d, file)):
        file, i = f"{stem}-{i}{ext}", i + 1
    open(os.path.join(d, file), "wb").write(raw)
    write_caption(name, file, req.caption or "")
    return {"file": file, "caption": read_caption(name, file)}

@app.delete("/training/dataset/{name}/image/{file}")
def delete_image(name: str, file: str):
    try:
        file = clean_file(file)
    except ValueError as e:
        fail(e)
    path = os.path.join(folder(name), file)
    if not os.path.exists(path):
        fail("Image not found", 404)
    os.remove(path)
    cap = caption_path(name, file)
    if os.path.exists(cap):
        os.remove(cap)
    return dataset_payload(name)

@app.post("/training/dataset/{name}/autotag")
def autotag(name: str, req: AutoTag):
    try:
        clean_name(name)
        _begin("auto-tagging", "Starting the tagger")
    except (ValueError, RuntimeError) as e:
        fail(e)
    threading.Thread(target=_worker, args=(name, float(req.threshold), req.mode or "replace"), daemon=True).start()
    return dict(STATE)

def serve():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

subprocess.run(["bash", "-lc", "fuser -k 8000/tcp >/dev/null 2>&1 || true"], check=False)
time.sleep(0.4)

threading.Thread(target=serve, daemon=True).start()
cf = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in cf.stdout:
    m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
if not url:
    raise RuntimeError("cloudflared did not print a URL — run this cell again")
print(f"READY: {url}")
print("Paste that into the Character Set tab. Leave this running while you drop pictures.")

